# Introduction: building and printing PDE problems

The Poisson problem from section 8 of the plan, written with `qewton.equations`. Everything is imported as `eq`.

In [ ]:
import qewton.equations as eq

## Coordinates and sets

A coordinate is a `Variable`. The domain Omega lives in `x`, and its boundary is split into two parts.

In [ ]:
x = eq.Variable('x', dim=2)
Omega = eq.Domain('Omega', ambient=x)
dO = eq.Boundary(Omega)
GD = eq.Part('Gamma_D', dO)
GN = eq.Part('Gamma_N', dO)

## Functions

Every function states its inputs. `None` declares a constant.

In [ ]:
u = eq.FunctionSymbol(eq.Variable('u'), x)        # unknown, depends on x
f = eq.FunctionSymbol(eq.Variable('f'), None)     # constant source
g = eq.FunctionSymbol(eq.Variable('g'), x)        # Dirichlet data on Gamma_D
h = eq.FunctionSymbol(eq.Variable('h'), x)        # Neumann data on Gamma_N
nu = eq.FunctionSymbol(eq.Variable('nu'), None)   # coefficient, constant in x

## Expressions

Operators come from `eq`. `print` shows them as math, and a bare expression shows the same text.

In [ ]:
flux = nu * eq.grad(u, x)
lhs = -eq.div(flux, x)
print(lhs)

In [ ]:
eq.dot(eq.Restrict(flux, GN), eq.Normal(GN))

## Constraints and the problem

`Eq(lhs, rhs).on(set)` attaches the set where the constraint holds. `Problem` refuses a constraint that has no set.

In [ ]:
problem = eq.Problem(
    constraints=(
        eq.Eq(lhs, f).on(Omega),
        eq.Eq(eq.Restrict(u, GD), g).on(GD),
        eq.Eq(eq.dot(eq.Restrict(flux, GN), eq.Normal(dO)), h).on(GN),
    ),
    partitions=(eq.Partition(dO, (GD, GN)),),
)
print(problem)

In [ ]:
print('functions:', sorted(s.name for s in problem.symbols()))
print('sets:', sorted(str(s) for s in problem.sets()))

### Overview

`problem.overview()` collects what the problem is made of: the coordinates (with the declared time marked), every set (including the factors of products), and every function with its inputs and output shape. A shape that was not declared and had to be assumed is marked. The pieces are also available one by one: `problem.symbols()`, `problem.sets()`, `problem.coordinates()` and `problem.time`.

In [ ]:
print(problem.overview())

## Time-dependent problems

The heat equation has a time variable. Declaring it with `time=` tells the analysis which variable evolves. The initial condition lives on the slice `t = 0`.

In [ ]:
t = eq.Variable('t', dim=1)
Interval = eq.Domain('I', ambient=t)
w = eq.FunctionSymbol(eq.Variable('w'), x * t)   # depends on x and t, in this order
w0 = eq.FunctionSymbol(eq.Variable('w0'), x)     # initial data
space_time = eq.Product((Omega, Interval))
initial = eq.Product((Omega, eq.Slice(t, 0.0)))
heat = eq.Problem(
    constraints=(
        eq.Eq(eq.dt(w, t) - eq.lap(w, x), f).on(space_time),
        eq.Eq(eq.Restrict(w, initial), w0).on(initial),
    ),
    time=t,
)
heat

In [ ]:
print(heat.overview())

## Next steps

The analysis passes (validation, shapes, classification) and the report come in the next milestones. This notebook only builds and prints problems.

## Integrals, delays and periodic conditions

- `eq.integral(u, S)` and `eq.mean(u, S)` integrate over a set.
- A constraint with `.on(None)` holds globally, e.g. a zero mean.
- `eq.shift(u, t, -1)` is `u(t − 1)`, for delays and periodic conditions.
- `eq.Periodic(a, b, x, offset)` declares that set `b` is set `a` moved by `offset`.

In [ ]:
from qewton.equations import fixtures
fixtures.neumann_poisson()

In [ ]:
fixtures.delay()

In [ ]:
fixtures.periodic_poisson()

## Tasks: what do we do with the problem?

A `Problem` is only mathematics. A *task* says what to do with it by giving each function a role:

- `unknown`: solve for it.
- `fixed`: one known value (a number, an expression, an array). Functions you do not list are fixed too, and a warning tells you when they have no value.
- `varying`: known per instance. The function gets its own batch axis `p_<name>`, one entry per instance.

Sets get a role too, but only when needed:

- A set is **fixed** by default. Pass it with a value, `fixed={Omega: mesh}`, when you give arrays that live on it (an array over `x` needs the points of `Omega`). Numbers and expressions need no mesh. If you forget one, `TaskError` names the missing set.
- A set is **varying** if it changes per instance, e.g. a moving boundary: `varying={Omega: meshes}` with one mesh per instance. Everything derived from it (its boundary, its parts, products with it) varies too.

Measurements are not a role: write them as an ordinary constraint.

In [ ]:
import numpy as np
mesh = np.random.rand(200, 2)      # stand-in for a mesh of Omega

**Forward solve.** One instance: everything is fixed except `u`.

In [ ]:
problem.symbols()

In [ ]:
forward = problem.task(
    unknown=[u],
    fixed={f: 1.0, nu: 1.0, g: 0.0, h: 0.0, Omega: mesh},
)
print(forward)
forward.diagnostics

**Parameter sweep.** 50 coefficients, one solution each. `nu_samples` has one entry per instance.

In [ ]:
nu_samples = np.linspace(0.5, 2.0, 50)
sweep = problem.task(
    unknown=[u],
    fixed={f: 1.0, g: 0.0, h: 0.0, Omega: mesh},
    varying={nu: nu_samples},
)
print(sweep)

**Operator learning.** The source is the input function and `u` the output. `f` is a constant here, so each sample is one number; a source that depends on `x` would be an array over the mesh.

In [ ]:
operator = problem.task(
    unknown=[u],
    fixed={nu: 1.0, g: 0.0, h: 0.0, Omega: mesh},
    varying={f: np.random.rand(100)},
)
print(operator)

**Inverse problem.** The coefficient is unknown as well. To give the solver something to learn from, add measurements of `u` as a constraint on a sensor set. The data function is fixed.

In [ ]:
sensors = eq.Part('Sensors', eq.Domain('Omega_s', ambient=x))
data = eq.FunctionSymbol(eq.Variable('data'), x)
inverse_problem = eq.Problem(
    constraints=problem.constraints
    + (eq.Eq(eq.Restrict(u, sensors), data).on(sensors),),
    partitions=problem.partitions,
)
inverse = inverse_problem.task(
    unknown=[u, nu],
    fixed={f: 1.0, g: 0.0, h: 0.0, Omega: mesh},
    varying={data : np.random.rand(30)},
)
print(inverse)

**Heat equation.** The initial condition varies per instance. Time needs no mesh, and the mesh of `Omega` serves `Omega × I`.

In [ ]:
heat_task = heat.task(
    unknown=[w],
    fixed={f: 0.0, Omega: mesh},
    varying={w0: np.random.rand(20, 200)},   # 20 initial conditions on 200 points
)
print(heat_task)

**Mistakes are caught when the task is built.**

In [ ]:
try:
    problem.task(unknown=[u], fixed=[u])
except eq.TaskError as error:
    print('TaskError:', error)
try:
    problem.task(unknown=[u], fixed={g: np.ones(10)})   # an array over x, but no mesh
except eq.TaskError as error:
    print('TaskError:', error)

**Varying domain.** The same Poisson problem on 10 different shapes of `Omega`. The boundary and its parts vary with it.

In [ ]:
shapes = problem.task(
    unknown=[u],
    fixed={f: 1.0, nu: 1.0, g: 0.0, h: 0.0},
    varying={Omega: np.random.rand(10, 200, 2)},
)
print(shapes)
print('boundary varies:', shapes.varies(dO), '| Gamma_D varies:', shapes.varies(GD))

## Normalizing

`eq.normalize` rewrites a problem into one canonical form. It returns a new problem and keeps the original constraints in `result.origin`; the problem you built is never changed. It does three things:

1. **Time lifting.** For a time-dependent function, a condition written without time means "at every time". Normalizing makes that explicit by moving it to `S × I`. Initial conditions are not lifted, and if no interval on the time coordinate is declared, nothing is lifted and a warning explains why.
2. **Laplacian.** `lap(u, x)` becomes `div(grad(u, x), x)`, so `-Δu` and `-div(∇u)` are analysed the same way.
3. **Residual form.** `lhs = rhs` becomes `lhs − rhs = 0`. A complementarity condition is left alone.

In [ ]:
gD = eq.FunctionSymbol(eq.Variable('gD', dim=1), x)
wt = eq.FunctionSymbol(eq.Variable('wt', dim=1), x * t)
heat_dirichlet = eq.Problem(
    constraints=(
        eq.Eq(eq.dt(wt, t) - eq.lap(wt, x), 0).on(space_time),
        eq.Eq(eq.Restrict(wt, dO), gD).on(dO),          # no time written
        eq.Eq(eq.Restrict(wt, initial), w0).on(initial),
    ),
    time=t,
)
result = eq.normalize(heat_dirichlet)
print(result.problem)
for d in result.diagnostics:
    print(d)

In [ ]:
print(eq.normalize(problem).problem)
print()
print('original:')
for c in eq.normalize(problem).origin:
    print(' ', c)

## Signatures

`eq.signatures(task)` lists every function the way a type checker would: its inputs, its output shape, its role from the task, and the sets it is evaluated on. A varying function gets its batch axis as an extra input. Inputs are never guessed; they are what you declared. The analysis adds the shape (marked when it was assumed), the sets, and a warning for a declared input that no constraint can use.

In [ ]:
print(eq.signatures(forward))
print()
print(eq.signatures(sweep))
print()
print(eq.signatures(heat_task))

In [ ]:
eq.signatures(forward).diagnostics

## Pipeline and report

`eq.run_pipeline(task)` runs validate, normalize and signatures in order and returns an `AnalysisContext` holding the task, the normalized problem, the signatures and all findings (errors first). `eq.report(ctx)` prints it as one text. `ctx.ok` is false if any finding is an error.

In [ ]:
ctx = eq.run_pipeline(forward)
print(eq.report(ctx))
ctx.ok

In [ ]:
print(eq.report(eq.run_pipeline(heat_task)))

## Structure: derivative orders

`unknown_orders(task)` gives, for each constraint, the highest derivative order of every unknown along each coordinate. Orders add up along the path to the function (`lap` is 2 in `x`), and a shift (delay) is not a derivative. This is the first part of the structure analysis; operator/source split, linearity and the coupling graph follow.

In [ ]:
from qewton.equations.analysis.structure import unknown_orders

for c, orders in zip(heat_task.problem.constraints, unknown_orders(heat_task)):
    print(c)
    for f, o in orders.items():
        print('   ', f.variable.name, {a.name: n for a, n in o.items()})

## Structure: operator and source

`split(task)` cuts each normalized constraint at its top-level sums. Terms that contain an unknown form the **operator** part, the others the **source** part. Each term is tagged `none`, `linear`, `affine` or `nonlinear` in the unknowns, using the operator registry: a product stays linear when only one factor contains unknowns, and is nonlinear when both do. A constraint is `linear` when its operator terms are all linear or affine. Signs are relative to the form `terms rel 0`.

In [ ]:
from qewton.equations.analysis.structure import split

for c, s in zip(eq.normalize(forward.problem).problem.constraints, split(forward)):
    print(c)
    print('  ', s.linearity)
    for t in s.operator:
        print('   operator', t)
    for t in s.source:
        print('   source  ', t)